# PART 04 · 3장 분산분석과 카이제곱 검정

교재 336~355쪽 본문 소스 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

#### 1. 평균 검정과 빈도 검정 구분

#### 2. 공통 라이브러리와 데이터 불러오기

다음 코드는 이 장에서 사용하는 라이브러리와 데이터를 준비한다.

In [1]:
import pandas as pd
from scipy import stats
rating_path = (
    'data/part04_type3/anova_support_channel_rating/'
    'support_rating.csv'
)
device_path = (
    'data/part04_type3/chisquare_device_plan/'
    'device_plan.csv'
)
rating_df = pd.read_csv(rating_path)
device_df = pd.read_csv(device_path)
print(rating_df.shape)
print(device_df.shape)

(128, 3)
(420, 3)


## 01 분산분석의 이해

### 1 사용 상황과 가설

#### 1. 사용하는 상황

#### 2. 귀무가설과 대립가설

### 2 F-statistic과 p-value

#### 1. F-statistic과 p-value

## 02 일원분산분석

### 1 집단별 요약 확인

#### 1. 집단별 요약 통계 확인

집단별 개수, 평균, 표준편차부터 확인해본다.

In [2]:
group_summary = (
    rating_df.groupby('support_channel')['satisfaction']
    .agg(['count', 'mean', 'std'])
    .round(4)
)
print(group_summary)

                 count    mean     std
support_channel                       
chat                32  4.0109  0.4198
email               32  3.5434  0.5487
phone               32  3.3378  0.4065
self_service        32  3.9269  0.5146


### 2 분산분석 실행

#### 1. SciPy로 일원분산분석 실행

집단별 데이터를 리스트로 만들어 SciPy로 일원분산분석을 실행해본다.

In [3]:
groups = [
    group['satisfaction']
    for _, group in rating_df.groupby('support_channel')
]
anova_result = stats.f_oneway(*groups)
print('f_statistic: ', round(anova_result.statistic, 4))
print('p_value: ', format(anova_result.pvalue, '.12f'))
alpha = 0.05
if anova_result.pvalue < alpha:
    print('decision: 귀무가설 기각')
else:
    print('decision: 귀무가설 기각하지 못함')

f_statistic:  14.2828
p_value:  0.000000047196
decision: 귀무가설 기각


#### 2. statsmodels 분산분석표 읽기

statsmodels를 사용하면 분산분석표 형태로 결과를 볼 수 있다.

In [4]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
model = ols(
    'satisfaction ~ C(support_channel)',
    data=rating_df
).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

                       sum_sq     df          F        PR(>F)
C(support_channel)   9.720165    3.0  14.282794  4.719601e-08
Residual            28.129428  124.0        NaN           NaN


## 03 사후검정

### 1 Tukey HSD 실행

#### 1. Tukey HSD 실행

분산분석에서 귀무가설이 기각되면 어느 집단 조합이 다른지 사후검정으로 확인한다. Tukey HSD를 실행해본다.

In [5]:
from itertools import combinations
from statsmodels.stats.multicomp import pairwise_tukeyhsd
posthoc = pairwise_tukeyhsd(
    endog=rating_df['satisfaction'],
    groups=rating_df['support_channel'],
    alpha=0.05
)
pairs = list(combinations(posthoc.groupsunique, 2))
posthoc_table = pd.DataFrame({
    'group1': [pair[0] for pair in pairs],
    'group2': [pair[1] for pair in pairs],
    'meandiff': posthoc.meandiffs,
    'p-adj': posthoc.pvalues,
    'lower': posthoc.confint[:, 0],
    'upper': posthoc.confint[:, 1],
    'reject': posthoc.reject
})
numeric_cols = ['meandiff', 'p-adj', 'lower', 'upper']
print(posthoc_table.to_string(
    index=False,
    formatters={
        col: '{:.4f}'.format
        for col in numeric_cols
    }
))

group1       group2 meandiff  p-adj   lower   upper  reject
  chat        email  -0.4675 0.0008 -0.7776 -0.1574    True
  chat        phone  -0.6731 0.0000 -0.9832 -0.3630    True
  chat self_service  -0.0841 0.8946 -0.3942  0.2260   False
 email        phone  -0.2056 0.3142 -0.5157  0.1045   False
 email self_service   0.3834 0.0088  0.0733  0.6935    True
 phone self_service   0.5891 0.0000  0.2790  0.8992    True

### 2 결과표 해석

#### 1. 결과표 읽기

#### 2. 유의한 집단 조합

## 04 카이제곱 적합도 검정

### 1 관측도수와 기대도수

#### 1. 관측도수와 기대도수

관측도수를 집계하고 모든 기기가 균등하다고 가정한 기대도수를 만들어 본다.

In [6]:
device_order = ['mobile', 'desktop', 'tablet']
observed = (
    device_df['device_type']
    .value_counts()
    .reindex(device_order)
)
expected = [len(device_df)/len(device_order)]*len(device_order)
print('observed: ', observed.to_dict())
print('expected: ', expected)

observed:  {'mobile': 180, 'desktop': 150, 'tablet': 90}
expected:  [140.0, 140.0, 140.0]


### 2 적합도 검정 실행

#### 1. 적합도 검정 실행

관측도수와 기대도수로 적합도 검정을 실행해본다.

In [7]:
gof_result = stats.chisquare(
    f_obs=observed.values,
    f_exp=expected
)
print('chi2_statistic: ', round(gof_result.statistic, 4))
print('p_value: ', format(gof_result.pvalue, '.9f'))
print('dof: ', len(device_order)-1)

chi2_statistic:  30.0
p_value:  0.000000306
dof:  2


#### 2. 범주별 카이제곱 기여도

## 05 카이제곱 독립성 검정

### 1 검정 구분과 가설

#### 1. 적합도 검정과 독립성 검정 구분

#### 2. 가설 설정

### 2 독립성 검정 실행

#### 1. 독립성 검정 실행

교차표를 만들어 독립성 검정을 실행해본다.

In [8]:
cross_tab = pd.crosstab(
    device_df['device_type'],
    device_df['plan_grade']
)
cross_tab = cross_tab.reindex(
    index=['mobile', 'desktop', 'tablet'],
    columns=['basic', 'standard', 'premium']
)
ind_result = stats.chi2_contingency(cross_tab)
print(cross_tab)
print('chi_square: ', round(ind_result.statistic, 4))
print('p_value: ', format(ind_result.pvalue, '.9f'))
print('dof: ', ind_result.dof)

plan_grade   basic  standard  premium
device_type                          
mobile          70        78       32
desktop         32        66       52
tablet          40        37       13
chi_square:  25.2197
p_value:  0.000045443
dof:  4


## 06 교차표 생성

### 1 교차표 만들기

#### 1. pd.crosstab( ) 기본 사용

pd.crosstab( )에 두 컬럼을 전달하면 교차표가 만들어진다.

In [9]:
cross_tab = pd.crosstab(
    device_df['device_type'],
    device_df['plan_grade']
)
print(cross_tab)

plan_grade   basic  premium  standard
device_type                          
desktop         32       52        66
mobile          70       32        78
tablet          40       13        37


#### 2. 행과 열 순서 고정

기본 교차표는 값의 알파벳 순서로 정렬된다. reindex로 행과 열 순서를 고정해본다.

In [10]:
cross_tab = cross_tab.reindex(
    index=['mobile', 'desktop', 'tablet'],
    columns=['basic', 'standard', 'premium']
)
print(cross_tab)

plan_grade   basic  standard  premium
device_type                          
mobile          70        78       32
desktop         32        66       52
tablet          40        37       13


### 2 빈도표와 비율표

#### 1. 빈도표와 비율표 구분

normalize 옵션을 사용하면 빈도 대신 비율을 확인할 수 있다.

In [11]:
row_ratio = pd.crosstab(
    device_df['device_type'],
    device_df['plan_grade'],
    normalize='index'
).round(4)
print(row_ratio)

plan_grade    basic  premium  standard
device_type                           
desktop      0.2133   0.3467    0.4400
mobile       0.3889   0.1778    0.4333
tablet       0.4444   0.1444    0.4111


## 07 관측도수와 기대도수

### 1 기대도수 계산

#### 1. 기대도수 계산

독립성 검정 결과에서 기대도수를 꺼내 관측도수와 나란히 비교해본다.

In [12]:
ind_result = stats.chi2_contingency(cross_tab)
expected_df = pd.DataFrame(
    ind_result.expected_freq,
    index=cross_tab.index,
    columns=cross_tab.columns
)
print('observed')
print(cross_tab)
print('expected')
print(expected_df.round(2))

observed
plan_grade   basic  standard  premium
device_type                          
mobile          70        78       32
desktop         32        66       52
tablet          40        37       13
expected
plan_grade   basic  standard  premium
device_type                          
mobile       60.86     77.57    41.57
desktop      50.71     64.64    34.64
tablet       30.43     38.79    20.79


#### 2. 기대도수 계산식

특정 셀의 기대도수는 해당 행의 합계와 열의 합계를 곱한 뒤 전체 합계로 나누어 계산한다. 결과적으로 각 셀의 기여도를 모두 더하면 카이제곱 검정통계량이 된다.

In [13]:
mobile_basic_expected = 180 * 142 / 420
print(round(mobile_basic_expected, 4))

60.8571


### 2 조건과 기여도

#### 1. 기대도수 조건과 셀별 기여도

기대도수 조건을 확인하고 셀별 카이제곱 기여도를 계산해본다.

In [14]:
contribution = (cross_tab - expected_df) ** 2 / expected_df
print('min_expected: ', round(expected_df.min().min(), 4))
print('expected_over_5: ', (expected_df >= 5).all().all())
print(contribution.round(4))
print('sum: ', round(contribution.to_numpy().sum(), 4))

min_expected:  20.7857
expected_over_5:  True
plan_grade    basic  standard  premium
device_type                           
mobile       1.3736    0.0024   2.2037
desktop      6.9058    0.0285   8.6965
tablet       3.0107    0.0822   2.9163
sum:  25.2197


## 08 검정통계량 해석

### 1 통계량의 의미

#### 1. 검정별 통계량의 의미

### 2 통계량 구성 확인

#### 1. 카이제곱 검정통계량과 자유도

독립성 검정 결과에서 검정통계량과 자유도를 꺼내 본다.

In [15]:
chi2_statistic = ind_result.statistic
p_value = ind_result.pvalue
dof = ind_result.dof
print('chi_square: ', round(chi2_statistic, 4))
print('p_value: ', format(p_value, '.9f'))
print('dof: ', dof)
print('contribution_sum: ', round(contribution.to_numpy().sum(), 4))

chi_square:  25.2197
p_value:  0.000045443
dof:  4
contribution_sum:  25.2197


#### 2. F-statistic의 구성

분산분석표의 값으로 F-statistic이 어떻게 만들어지는지 직접 계산해본다.

In [16]:
between_ms = 9.720165 / 3
within_ms = 28.129428/124
f_statistic = between_ms/within_ms
print('between_ms: ', round(between_ms, 4))
print('within_ms: ', round(within_ms, 4))
print('f_statistic: ', round(f_statistic, 4))

between_ms:  3.2401
within_ms:  0.2269
f_statistic:  14.2828


## 09 p-value 해석

### 1 귀무가설 확인

#### 1. 검정별 귀무가설 확인

### 2 결과 비교와 표기

#### 1. 세 가지 결과 비교

세 가지 검정의 p-value를 같은 기준으로 판단해본다.

In [17]:
alpha = 0.05
levene_result = stats.levene(*groups)
results = [
    ('일원분산분석', anova_result.pvalue),
    ('카이제곱 독립성 검정', ind_result.pvalue),
    ('등분산성 검정', levene_result.pvalue),
]
for name, p_value in results:
    if p_value < alpha:
        decision = '귀무가설 기각'
    else:
        decision = '귀무가설 기각하지 못함'
    print(name, format(p_value, '.12f'), decision)

일원분산분석 0.000000047196 귀무가설 기각
카이제곱 독립성 검정 0.000045442563 귀무가설 기각
등분산성 검정 0.253198401911 귀무가설 기각하지 못함


#### 2. 매우 작은 p-value 표기

## 10 가설 기각 여부 판단

### 1 판단 순서

#### 1. 판단의 기본 순서

### 2 검정별 결론 정리

#### 1. 네 가지 검정 결과를 한 번에 판단하기

네 가지 검정 결과를 같은 판단 함수로 한 번에 처리해본다.

In [18]:
alpha = 0.05
def decision_text(p_value):
    if p_value < alpha:
        return '귀무가설 기각'
    return '귀무가설 기각하지 못함'
def exam_text(p_value):
    if p_value < alpha:
        return '귀무가설 기각'
    return '귀무가설을 기각하지 못함'
results = [
    ('일원분산분석', anova_result),
    ('등분산성 검정', levene_result),
    ('카이제곱 독립성 검정', ind_result),
    ('카이제곱 적합도 검정', gof_result),
]
for name, result in results:
    print(name)
    print('statistic: ', round(result.statistic, 4))
    print('p_value: ', format(result.pvalue, '.9f'))
    print('decision: ', decision_text(result.pvalue))
    print('exam_answer: ', exam_text(result.pvalue))
    print()

일원분산분석
statistic:  14.2828
p_value:  0.000000047
decision:  귀무가설 기각
exam_answer:  귀무가설 기각

등분산성 검정
statistic:  1.3762
p_value:  0.253198402
decision:  귀무가설 기각하지 못함
exam_answer:  귀무가설을 기각하지 못함

카이제곱 독립성 검정
statistic:  25.2197
p_value:  0.000045443
decision:  귀무가설 기각
exam_answer:  귀무가설 기각

카이제곱 적합도 검정
statistic:  30.0
p_value:  0.000000306
decision:  귀무가설 기각
exam_answer:  귀무가설 기각



#### 2. 검정별 결론 문장

#### 3. 채택과 기각하지 못함의 차이